# 05 — V2 Multi-Seed Robustness Evaluation

## Purpose

The initial V2 result came from one random seed. This notebook repeats the **same local-only and federated V2 architecture** across five seeds to test whether the observed differences are stable rather than training randomness.

Seeds: **42, 52, 62, 72, 82**

The architecture is not changed in this experiment.

For every seed:
- the original V1 held-out test split remains fixed (`random_state=42`);
- the validation split also remains fixed;
- only neural-network initialisation, DataLoader shuffling and training randomness change;
- local-only and federated V2 models are trained;
- thresholds are selected from validation data only;
- the untouched test set is evaluated.

The main comparison is **V2 Local vs V2 Federated** on each dataset.


In [1]:
%pip install scikit-learn torch

import os, copy, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

SPLIT_SEED = 42
SEEDS = [42, 52, 62, 72, 82]
LATENT_DIM = 8

print("Seeds:", SEEDS)
print("PyTorch:", torch.__version__)


Note: you may need to restart the kernel to use updated packages.
Seeds: [42, 52, 62, 72, 82]
PyTorch: 2.2.2


## 1. Load data and reproduce the fixed V1 test sets


In [2]:
DATASET1_PATH = "../data/diabetes_prediction_dataset.csv"
DATASET2_PATH = "../data/diabetes_dataset.csv"

df1 = pd.read_csv(DATASET1_PATH)
df2 = pd.read_csv(DATASET2_PATH)

TARGET1 = "diabetes"
TARGET2 = "diagnosed_diabetes"

df2_model = df2.drop(
    columns=[c for c in ["diabetes_stage", "diabetes_risk_score"] if c in df2.columns]
).copy()

X1, y1 = df1.drop(columns=[TARGET1]), df1[TARGET1].astype(int)
X2, y2 = df2_model.drop(columns=[TARGET2]), df2_model[TARGET2].astype(int)

# Fixed outer split: same held-out examples as V1/V2.
X1_train_full, X1_test, y1_train_full, y1_test = train_test_split(
    X1, y1, test_size=0.20, random_state=SPLIT_SEED, stratify=y1
)
X2_train_full, X2_test, y2_train_full, y2_test = train_test_split(
    X2, y2, test_size=0.20, random_state=SPLIT_SEED, stratify=y2
)

# Fixed validation split. Only model-training randomness varies across runs.
X1_train, X1_val, y1_train, y1_val = train_test_split(
    X1_train_full, y1_train_full,
    test_size=0.125, random_state=SPLIT_SEED, stratify=y1_train_full
)
X2_train, X2_val, y2_train, y2_val = train_test_split(
    X2_train_full, y2_train_full,
    test_size=0.125, random_state=SPLIT_SEED, stratify=y2_train_full
)

print("D1 train/val/test:", len(X1_train), len(X1_val), len(X1_test))
print("D2 train/val/test:", len(X2_train), len(X2_val), len(X2_test))


D1 train/val/test: 70000 10000 20000
D2 train/val/test: 70000 10000 20000


## 2. Fit client-specific preprocessing once on the fixed training sets


In [3]:
def build_preprocessor(X):
    cat = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    num = [c for c in X.columns if c not in cat]

    return ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat)
    ])

prep1 = build_preprocessor(X1_train)
prep2 = build_preprocessor(X2_train)

A_train = prep1.fit_transform(X1_train).astype(np.float32)
A_val = prep1.transform(X1_val).astype(np.float32)
A_test = prep1.transform(X1_test).astype(np.float32)

B_train = prep2.fit_transform(X2_train).astype(np.float32)
B_val = prep2.transform(X2_val).astype(np.float32)
B_test = prep2.transform(X2_test).astype(np.float32)

print("Client A input dim:", A_train.shape[1])
print("Client B input dim:", B_train.shape[1])


Client A input dim: 15
Client B input dim: 46


## 3. Same V2 architecture and training functions


In [4]:
class ClientEncoder(nn.Module):
    def __init__(self, input_dim, latent_dim=LATENT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, latent_dim),
            nn.ReLU()
        )
    def forward(self, x):
        return self.net(x)

class SharedPredictor(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )
    def forward(self, z):
        return self.net(z).squeeze(1)

class ClientModel(nn.Module):
    def __init__(self, encoder, predictor):
        super().__init__()
        self.encoder = encoder
        self.predictor = predictor
    def forward(self, x):
        return self.predictor(self.encoder(x))

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def make_loader(X, y, batch_size=512, shuffle=True):
    return DataLoader(
        TensorDataset(
            torch.tensor(X, dtype=torch.float32),
            torch.tensor(np.asarray(y), dtype=torch.float32)
        ),
        batch_size=batch_size,
        shuffle=shuffle
    )

def train_model(model, X, y, epochs=5, lr=1e-3, batch_size=512):
    loader = make_loader(X, y, batch_size, True)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    model.train()

    for _ in range(epochs):
        for xb, yb in loader:
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
    return model

def predict_prob(model, X):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(
            model(torch.tensor(X, dtype=torch.float32))
        ).cpu().numpy()

def best_f1_threshold(y_true, prob):
    thresholds = np.linspace(0.05, 0.95, 181)
    scores = [
        f1_score(y_true, (prob >= t).astype(int), zero_division=0)
        for t in thresholds
    ]
    return float(thresholds[int(np.argmax(scores))])

def metrics(y_true, prob, threshold):
    pred = (prob >= threshold).astype(int)
    return {
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, prob)
    }

def weighted_average_predictors(predictors, sample_counts):
    total = float(sum(sample_counts))
    state = copy.deepcopy(predictors[0].state_dict())
    for key in state:
        state[key] = torch.zeros_like(state[key])
        for p, n in zip(predictors, sample_counts):
            state[key] += p.state_dict()[key] * (n / total)
    result = SharedPredictor()
    result.load_state_dict(state)
    return result


## 4. One complete local-vs-federated run


In [5]:
def run_seed(seed, rounds=5, local_epochs=3):
    set_seed(seed)

    # LOCAL-ONLY CONTROLS: same as V2, 20 epochs.
    local_A = ClientModel(
        ClientEncoder(A_train.shape[1]), SharedPredictor()
    )
    local_B = ClientModel(
        ClientEncoder(B_train.shape[1]), SharedPredictor()
    )
    train_model(local_A, A_train, y1_train, epochs=20)
    train_model(local_B, B_train, y2_train, epochs=20)

    # FEDERATED V2
    # Reset seed so each run is reproducible for the requested training seed.
    set_seed(seed)
    enc_A = ClientEncoder(A_train.shape[1])
    enc_B = ClientEncoder(B_train.shape[1])
    global_pred = SharedPredictor()

    for _ in range(rounds):
        pred_A = copy.deepcopy(global_pred)
        pred_B = copy.deepcopy(global_pred)

        fed_A_round = ClientModel(enc_A, pred_A)
        fed_B_round = ClientModel(enc_B, pred_B)

        train_model(fed_A_round, A_train, y1_train, epochs=local_epochs)
        train_model(fed_B_round, B_train, y2_train, epochs=local_epochs)

        enc_A = fed_A_round.encoder
        enc_B = fed_B_round.encoder

        global_pred = weighted_average_predictors(
            [fed_A_round.predictor, fed_B_round.predictor],
            [len(A_train), len(B_train)]
        )

    fed_A = ClientModel(enc_A, copy.deepcopy(global_pred))
    fed_B = ClientModel(enc_B, copy.deepcopy(global_pred))

    models = [
        ("Local", "Dataset 1", local_A, A_val, y1_val, A_test, y1_test),
        ("Local", "Dataset 2", local_B, B_val, y2_val, B_test, y2_test),
        ("Federated", "Dataset 1", fed_A, A_val, y1_val, A_test, y1_test),
        ("Federated", "Dataset 2", fed_B, B_val, y2_val, B_test, y2_test),
    ]

    rows = []
    for approach, dataset, model, Xv, yv, Xt, yt in models:
        val_prob = predict_prob(model, Xv)
        threshold = best_f1_threshold(yv, val_prob)
        test_prob = predict_prob(model, Xt)

        row = {
            "Seed": seed,
            "Approach": approach,
            "Dataset": dataset,
            **metrics(yt, test_prob, threshold)
        }
        rows.append(row)

    return rows


## 5. Run all five seeds


In [6]:
all_rows = []

for i, seed in enumerate(SEEDS, start=1):
    print(f"Running seed {seed} ({i}/{len(SEEDS)})...")
    seed_rows = run_seed(seed)
    all_rows.extend(seed_rows)
    print(f"Completed seed {seed}")

multiseed_results = pd.DataFrame(all_rows)

for c in ["Threshold", "Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]:
    multiseed_results[c] = multiseed_results[c].astype(float)

multiseed_results.round(4)


Running seed 42 (1/5)...
Completed seed 42
Running seed 52 (2/5)...
Completed seed 52
Running seed 62 (3/5)...
Completed seed 62
Running seed 72 (4/5)...
Completed seed 72
Running seed 82 (5/5)...
Completed seed 82


,Seed,Approach,Dataset,Threshold,Accuracy,Precision,Recall,F1,ROC-AUC
0,42,Local,Dataset 1,0.415,0.9698,0.9329,0.6953,0.7968,0.9758
1,42,Local,Dataset 2,0.355,0.9096,0.9784,0.8684,0.9201,0.9416
2,42,Federated,Dataset 1,0.870,0.9678,0.9483,0.6576,0.7767,0.9636
3,42,Federated,Dataset 2,0.395,0.9114,0.9885,0.8622,0.9211,0.9421
4,52,Local,Dataset 1,0.470,0.9684,0.9428,0.6694,0.7829,0.9747
5,52,Local,Dataset 2,0.455,0.9098,0.9879,0.8601,0.9196,0.9420
6,52,Federated,Dataset 1,0.810,0.9703,0.9784,0.6653,0.7920,0.9709
7,52,Federated,Dataset 2,0.365,0.9113,0.9895,0.8613,0.9210,0.9415
8,62,Local,Dataset 1,0.460,0.9700,0.9568,0.6776,0.7934,0.9732
9,62,Local,Dataset 2,0.570,0.9137,0.9936,0.8618,0.9230,0.9419


## 6. Mean ± standard deviation across seeds


In [7]:
metric_cols = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]

summary = (
    multiseed_results
    .groupby(["Dataset", "Approach"])[metric_cols]
    .agg(["mean", "std"])
)

summary.round(4)


Accuracy         Precision          Recall          \
                        mean     std      mean     std    mean     std   
Dataset   Approach                                                       
Dataset 1 Federated   0.9683  0.0037    0.9414  0.0443  0.6700  0.0164   
          Local       0.9688  0.0020    0.9379  0.0212  0.6775  0.0122   
Dataset 2 Federated   0.9120  0.0007    0.9892  0.0029  0.8627  0.0024   
          Local       0.9112  0.0017    0.9883  0.0060  0.8623  0.0035   

                         F1         ROC-AUC          
                       mean     std    mean     std  
Dataset   Approach                                   
Dataset 1 Federated  0.7826  0.0229  0.9691  0.0036  
          Local      0.7867  0.0131  0.9736  0.0018  
Dataset 2 Federated  0.9216  0.0006  0.9418  0.0003  
          Local      0.9210  0.0013  0.9417  0.0003

In [8]:
# Easier-to-read formatted summary for the report / supervisor update.
formatted_rows = []

for (dataset, approach), group in multiseed_results.groupby(["Dataset", "Approach"]):
    row = {"Dataset": dataset, "Approach": approach}
    for m in metric_cols:
        row[m] = f"{group[m].mean():.4f} ± {group[m].std(ddof=1):.4f}"
    formatted_rows.append(row)

formatted_summary = pd.DataFrame(formatted_rows)
formatted_summary


,Dataset,Approach,Accuracy,Precision,Recall,F1,ROC-AUC
0,Dataset 1,Federated,0.9683 ± 0.0037,0.9414 ± 0.0443,0.6700 ± 0.0164,0.7826 ± 0.0229,0.9691 ± 0.0036
1,Dataset 1,Local,0.9688 ± 0.0020,0.9379 ± 0.0212,0.6775 ± 0.0122,0.7867 ± 0.0131,0.9736 ± 0.0018
2,Dataset 2,Federated,0.9120 ± 0.0007,0.9892 ± 0.0029,0.8627 ± 0.0024,0.9216 ± 0.0006,0.9418 ± 0.0003
3,Dataset 2,Local,0.9112 ± 0.0017,0.9883 ± 0.0060,0.8623 ± 0.0035,0.9210 ± 0.0013,0.9417 ± 0.0003


## 7. Paired federated-minus-local differences

Because local and federated runs use the same set of seeds, this table reports the per-seed difference and its mean.

- Positive = federated result is higher.
- Negative = federated result is lower.

With only five seeds, treat this as a robustness/descriptive experiment rather than making strong statistical-significance claims.


In [9]:
paired = (
    multiseed_results
    .pivot(index=["Seed", "Dataset"], columns="Approach", values=metric_cols)
)

delta_rows = []
for seed in SEEDS:
    for dataset in ["Dataset 1", "Dataset 2"]:
        row = {"Seed": seed, "Dataset": dataset}
        for m in metric_cols:
            row[f"Delta {m}"] = (
                paired.loc[(seed, dataset), (m, "Federated")]
                - paired.loc[(seed, dataset), (m, "Local")]
            )
        delta_rows.append(row)

paired_deltas = pd.DataFrame(delta_rows)
paired_deltas.round(4)


,Seed,Dataset,Delta Accuracy,Delta Precision,Delta Recall,Delta F1,Delta ROC-AUC
0,42,Dataset 1,-0.0020,0.0153,-0.0376,-0.0201,-0.0122
1,42,Dataset 2,0.0018,0.0101,-0.0062,0.0009,0.0005
2,52,Dataset 1,0.0019,0.0355,-0.0041,0.0091,-0.0038
3,52,Dataset 2,0.0016,0.0015,0.0012,0.0014,-0.0005
4,62,Dataset 1,-0.0080,-0.0902,-0.0241,-0.0483,-0.0059
5,62,Dataset 2,-0.0015,-0.0087,0.0052,-0.0008,0.0001
6,72,Dataset 1,0.0049,0.0393,0.0288,0.0332,0.0002
7,72,Dataset 2,0.0009,0.0008,0.0007,0.0008,0.0001
8,82,Dataset 1,0.0010,0.0175,-0.0006,0.0056,-0.0009
9,82,Dataset 2,0.0010,0.0008,0.0011,0.0010,0.0003


In [10]:
delta_summary = (
    paired_deltas
    .groupby("Dataset")[[c for c in paired_deltas.columns if c.startswith("Delta")]]
    .agg(["mean", "std"])
)

delta_summary.round(4)


Delta Accuracy         Delta Precision         Delta Recall          \
                    mean     std            mean     std         mean     std   
Dataset                                                                         
Dataset 1        -0.0004  0.0049          0.0035  0.0534      -0.0075  0.0253   
Dataset 2         0.0007  0.0013          0.0009  0.0067       0.0004  0.0041   

          Delta F1         Delta ROC-AUC          
              mean     std          mean     std  
Dataset                                           
Dataset 1  -0.0041  0.0311       -0.0045  0.0049  
Dataset 2   0.0006  0.0009        0.0001  0.0003

## 8. Save robustness results


In [11]:
os.makedirs("../models", exist_ok=True)

multiseed_results.to_csv(
    "../models/prototype_v2_multiseed_results.csv", index=False
)
formatted_summary.to_csv(
    "../models/prototype_v2_multiseed_summary.csv", index=False
)
paired_deltas.to_csv(
    "../models/prototype_v2_multiseed_deltas.csv", index=False
)

print("Saved:")
print("../models/prototype_v2_multiseed_results.csv")
print("../models/prototype_v2_multiseed_summary.csv")
print("../models/prototype_v2_multiseed_deltas.csv")


Saved:
../models/prototype_v2_multiseed_results.csv
../models/prototype_v2_multiseed_summary.csv
../models/prototype_v2_multiseed_deltas.csv
